# NB06 — pathway and TF activity

**In:** TCGA intrinsic expression (METABRIC is not deconvolved)
**Out:** `pathway_activity.parquet`, `tf_activity.parquet`, `tf_reliability.parquet`
**Gate:** ER+ vs Estrogen pathway AUROC ≥ 0.8
**Control:** raw undeconvolved METABRIC microarray vs ER_IHC. Deconvolution should
improve ER separation; if bulk beats intrinsic, NB02 is destroying signal.


In [ ]:
from pathlib import Path
import sys, json, warnings
warnings.filterwarnings("ignore")

cwd = Path.cwd().resolve()
for cand in [cwd, *cwd.parents]:
    if (cand / "src" / "gate.py").is_file():
        sys.path.insert(0, str(cand / "src"))
        break
    nested = cand / "v2"
    if (nested / "src" / "gate.py").is_file():
        sys.path.insert(0, str(nested / "src"))
        break

from paths import ensure_src_on_path, resolve_v2_root
from gate import gate as _gate_impl
from safety import assert_safe

V2_ROOT = resolve_v2_root()
ensure_src_on_path(V2_ROOT)
REPO_ROOT = V2_ROOT.parent
RAW = V2_ROOT / "data" / "raw"
INTERIM = V2_ROOT / "data" / "interim"
REF = V2_ROOT / "data" / "reference"
ARTIFACTS = V2_ROOT / "artifacts"
FIGURES = V2_ROOT / "reports" / "figures"
for d in (RAW, INTERIM, REF, ARTIFACTS, FIGURES, INTERIM / "causal_networks"):
    d.mkdir(parents=True, exist_ok=True)

# Laptop vs VPS. Smoke passes are provisional until a full run converts them.
# NB01 and NB04 stay full: harmonisation and the VAE are cheap.
SMOKE_TEST = True
N_SAMPLES  = 200    if SMOKE_TEST else None   # NB02 bulk (BayesPrism; memory)
N_SC_CELLS = 25_000 if SMOKE_TEST else None   # NB02 Wu reference (BayesPrism; memory)
N_PATIENTS = 50     if SMOKE_TEST else None   # NB07 CARNIVAL (throughput, not RAM)
N_DRUGS    = 10     if SMOKE_TEST else None   # NB10 ODE (FLOPs, not RAM)

def gate(*args, **kwargs):
    kwargs.setdefault("smoke_test", SMOKE_TEST)
    return _gate_impl(*args, **kwargs)

print("V2_ROOT =", V2_ROOT, "SMOKE_TEST =", SMOKE_TEST)


In [ ]:
# Config
AUROC_MIN = 0.8
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score


In [ ]:
# Load
from io_data import encode_er_status
from deconv import read_cbioportal_matrix
intr = INTERIM / "intrinsic_expression.parquet"
expr = pd.read_parquet(intr) if intr.exists() else None
expr_src = "intrinsic_tcga" if expr is not None else "missing"
mb_clin = None
p = next((RAW / "metabric").rglob("*clinical_patient.txt"), None)
if p is not None:
    mb_clin = pd.read_csv(p, sep="\t", comment="#")
    print("METABRIC clinical", p)
tcga_clin = None
p = next((RAW / "tcga_brca").rglob("*clinical_patient.txt"), None)
if p is not None:
    tcga_clin = pd.read_csv(p, sep="\t", comment="#")
    print("TCGA clinical", p)
mb_bulk = None
m = next((RAW / "metabric").rglob("data_mrna_illumina_microarray.txt"), None)
if m is not None:
    mb_bulk = read_cbioportal_matrix(m)
    print("METABRIC bulk microarray", mb_bulk.shape, "median", float(np.nanmedian(mb_bulk.to_numpy())))
print("expression source", expr_src, None if expr is None else expr.shape)


In [ ]:
# Compute — METABRIC bulk control first, then TCGA intrinsic (the gate)
auroc = 0.0
note = "no expression"
n_er = 0
thin = False
net = tf_net = None
try:
    import decoupler as dc
    net = dc.op.progeny(organism="human", top=500)
    tf_net = dc.op.collectri(organism="human")
except Exception as e:
    print("decoupler priors failed", e)

def _pathway(mat):
    X = mat.select_dtypes(include=[np.number]).fillna(0)
    X.columns = X.columns.astype(str).str.upper()
    X.index = X.index.astype(str)
    if net is None:
        estrogen_genes = [g for g in ["ESR1","PGR","FOXA1","GATA3","TFF1","GREB1","CCND1"] if g in X.columns]
        pw = pd.DataFrame({"Estrogen": X[estrogen_genes].mean(1) if estrogen_genes else X.mean(1)}, index=X.index)
        return pw, pw.rename(columns={"Estrogen": "ESR1"})
    res = dc.mt.mlm(X, net)
    pw = pd.DataFrame(res[0] if isinstance(res, tuple) else res)
    if list(pw.index) != list(X.index) and list(pw.columns) == list(X.index):
        pw = pw.T
    pw.index = X.index
    tf = pw
    if tf_net is not None:
        tf_res = dc.mt.ulm(X, tf_net)
        tf = pd.DataFrame(tf_res[0] if isinstance(tf_res, tuple) else tf_res)
        if list(tf.index) != list(X.index) and list(tf.columns) == list(X.index):
            tf = tf.T
        tf.index = X.index
    return pw, tf

def _auroc(pw, y, label):
    est = next((c for c in pw.columns if str(c).lower()=="estrogen"), None)
    if est is None or y is None:
        print(label, "skip")
        return None
    y = y.dropna()
    common = pw.index.intersection(y.index)
    if len(common) < 20 or y.loc[common].nunique() < 2:
        print(label, "insufficient", len(common))
        return None
    s = pw.loc[common, est]
    yy = y.loc[common]
    val = float(roc_auc_score(yy, s))
    print(f"{label:42s} AUROC={val:.4f} n={len(common)} meanER+={float(s[yy==1].mean()):.3f} meanER-={float(s[yy==0].mean()):.3f}")
    return {"auroc": val, "n": int(len(common)), "mu_pos": float(s[yy==1].mean()), "mu_neg": float(s[yy==0].mean())}

controls = {}
if mb_bulk is not None and mb_clin is not None and "ER_IHC" in mb_clin.columns:
    idx = mb_clin.set_index("PATIENT_ID")
    idx.index = idx.index.astype(str)
    y_mb = encode_er_status(idx["ER_IHC"])
    pw_mb, _ = _pathway(mb_bulk)
    controls["metabric_bulk_ihc"] = _auroc(pw_mb, y_mb, "METABRIC bulk microarray vs ER_IHC")

y_tcga = None
er_col_used = "SUBTYPE_Lum_vs_Basal"
if tcga_clin is not None and "SUBTYPE" in tcga_clin.columns:
    tci = tcga_clin.set_index("PATIENT_ID")
    tci.index = tci.index.astype(str).str[:12]
    sub = tci["SUBTYPE"].astype(str)
    y_tcga = pd.Series(np.nan, index=sub.index)
    y_tcga[sub.str.contains("LumA|LumB", case=False, na=False)] = 1.0
    y_tcga[sub.str.contains("Basal", case=False, na=False)] = 0.0
    print("TCGA PAM50 lum vs basal", y_tcga.value_counts(dropna=False).to_dict())

if expr is not None:
    mat = expr.select_dtypes(include=[np.number]).fillna(0)
    mat.columns = mat.columns.astype(str).str.upper()
    mat.index = mat.index.astype(str).str[:12]
    pathway, tf = _pathway(mat)
    tf = tf.replace([np.inf, -np.inf], np.nan)
    pathway.to_parquet(INTERIM / "pathway_activity.parquet")
    tf.to_parquet(INTERIM / "tf_activity.parquet")
    rel = pd.DataFrame({"tf": list(tf.columns), "reliability": tf.notna().mean().to_numpy()})
    rel.to_parquet(INTERIM / "tf_reliability.parquet")
    hit = _auroc(pathway, y_tcga, "TCGA intrinsic vs PAM50 Lum/Basal")
    if hit is None:
        note = "insufficient ER labels on TCGA intrinsic"
        thin = True
    else:
        auroc, n_er = hit["auroc"], hit["n"]
        note = (f"n={n_er} source={expr_src} er_col={er_col_used} "
                f"mean_Estrogen ER+={hit['mu_pos']:.3f} ER-={hit['mu_neg']:.3f}")
        bulk_hit = controls.get("metabric_bulk_ihc")
        if bulk_hit:
            note += (f" | METABRIC_bulk_IHC AUROC={bulk_hit['auroc']:.3f} n={bulk_hit['n']} "
                     f"(deconv should beat bulk; bulk>intrinsic means NB02 destroyed signal)")
            if bulk_hit["auroc"] > auroc + 0.02:
                note += " | bulk_beats_intrinsic"
else:
    thin = True
(INTERIM / "NB06_bulk_vs_intrinsic.json").write_text(json.dumps({"gate": {"auroc": auroc, "n": n_er, "note": note}, "controls": controls}, default=str, indent=2))
print("AUROC", auroc, note)


In [ ]:
# GATE
gate("NB06", "estrogen_er_positive_control", float(auroc), AUROC_MIN,
     n=n_er, min_n=20, insufficient_data=thin, note=note)


In [ ]:
# Figures
try:
    import matplotlib.pyplot as plt
    p = INTERIM / "pathway_activity.parquet"
    if p.exists():
        pw = pd.read_parquet(p)
        fig, ax = plt.subplots(figsize=(5, 3))
        ax.hist(pw.iloc[:, 0], bins=30)
        ax.set_title(str(pw.columns[0]))
        fig.tight_layout(); fig.savefig(FIGURES / "NB06_estrogen.png", dpi=140)
except Exception as e:
    print(e)
